# Lab 6: Correlation and the Regression Line

<img src="ds4all-logo.png" alt="Data Science for All Logo" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

Welcome to Lab 6 of *Data Science for All*!

In lecture you measured how closely a scatter plot clusters around a line with the correlation $r$,
used $r$ to find the regression line, and measured the line's error with the RMSE. In this lab you'll
use all three to predict how long you'll wait for Old Faithful, a geyser in Yellowstone National Park,
to erupt again.

This lab will be due on **Thursday at 5 PM (Week 13)**. As always, collaborating on labs is encouraged,
but don't just share answers.

## Today's lab

In today's lab, you'll learn how to:

1. check whether two variables are roughly linearly related, and measure how closely with the correlation $r$;
2. find the regression line from $r$, the SDs, and the averages, and draw it with `geom_abline()`;
3. make predictions with the line, and measure how far off they are with residuals and the RMSE; and
4. spot predictions you shouldn't trust, and see whether two lines predict better than one.

## Resources

Keep these open while you work:

- **Reading:** from the Data 8 textbook, [15.1 Correlation](https://dats.columbian.gwu.edu/ds4all/modules/prediction.html#correlation), [15.2 The Regression Line](https://dats.columbian.gwu.edu/ds4all/modules/prediction.html#the-regression-line), [14.2 Variability](https://dats.columbian.gwu.edu/ds4all/modules/mean.html#variability) (for standard units), and [15.3 The Method of Least Squares](https://dats.columbian.gwu.edu/ds4all/modules/prediction.html#root-mean-squared-error) (for the RMSE)

These are the tools you'll use most:

| If you want to... | Use |
|---|---|
| convert a vector to standard units | `standard_units()`, which you'll write in Question 1.2 |
| find the correlation of two columns | `cor(x, y)` |
| add columns to a tibble | `mutate()` |
| keep the rows that meet a condition | `filter()` |
| draw a line with a given slope and intercept | `geom_abline(slope = ..., intercept = ...)` |
| pick a value row by row, based on a condition | `case_when()` |

In [ ]:
# Initialize the autograder and JupyterHub helper
source("helpers.R")

library(dplyr)
library(ggplot2)

options(repr.matrix.max.rows = 20)
options(repr.plot.width = 7, repr.plot.height = 4.5)

As in Labs 4 and 5, we load only packages that Coding Assignment 5's autograder has, so anything that
runs here will also run there.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 1. How Faithful is Old Faithful?

Old Faithful is a geyser in Yellowstone National Park that is famous for erupting on a fairly regular
schedule. [Watch it erupt](https://www.youtube.com/watch?v=wE8NDuzt8eg).

Some of Old Faithful's eruptions last longer than others. Whenever there is a long eruption, it is
usually followed by an even longer wait before the next eruption. If you visit Yellowstone, you might
want to predict when the next eruption will happen, so that you can see the rest of the park instead of
waiting by the geyser.

Today, we will use data on eruption durations and waiting times to see if we can make such predictions
accurately with linear regression.

---

**Question 1.0.** The following statements are the unordered steps of linear regression.

1. Compute the parameters of the regression line: the slope and the intercept.
2. Evaluate the regression line by computing the line's RMSE and analyzing the residual plot.
3. Use the regression line to generate predictions for each x value.
4. Determine if linear regression is a reasonable method by visualizing your data and computing the
   correlation coefficient.

Make a vector called `least_squares_order` with `c()` that lists the step numbers in the order of a
linear regression analysis: the first item is the first step, and the last item is the last step.

In [ ]:
least_squares_order <- NULL # YOUR CODE HERE

least_squares_order

In [ ]:
. <- ottr::check("tests/q1_0.R")

This lab follows those steps. R comes with the Old Faithful data, in the built-in dataset `faithful`.
It has one row for each of 272 observed eruptions, and two columns:

- `eruptions`: how long the eruption lasted, in minutes
- `waiting`: the time from this eruption to the next one, in minutes

The next cell turns it into a tibble, like the data you've read with `read_csv()`. Run `?faithful` if
you want to see its help page.

In [ ]:
faithful <- as_tibble(faithful)

faithful

We would like to use linear regression to make predictions, but that won't work well if the data
aren't roughly linearly related. To check that, we should look at the data.

---

**Question 1.1.** Make a scatter plot of the data, named `faithful_scatter`. It's conventional to put
the column we want to predict on the vertical axis and the other column on the horizontal axis. Fill in
the blanks, and label the axes in plain words, with units.

In [ ]:
faithful_scatter <- ggplot(faithful) +
  aes(x = ..., y = ...) +
  geom_...(alpha = 0.5) +
  labs(
    title = "...",
    x = "...",
    y = "..."
  ) +
  theme_minimal(base_size = 16)

faithful_scatter

Based on your scatter plot, how are eruption duration and waiting time related?

- `A.` Roughly linear and positive: longer eruptions tend to be followed by longer waits.
- `B.` Roughly linear and negative: longer eruptions tend to be followed by shorter waits.
- `C.` Not linear: the points form two clusters, so a line can't describe them.
- `D.` No association: the wait doesn't depend on how long the eruption lasted.

Set `shape_answer` to the letter, as a string.

In [ ]:
shape_answer <- NULL # YOUR CODE HERE

shape_answer

In [ ]:
. <- ottr::check("tests/q1_1.R")

The eruptions fall into two clusters: a bunch of short ones and a bunch of longer ones. But the points
in both clusters fall roughly along one line, and that's what matters for predicting waiting times with
linear regression. We'll come back to the clusters in Part 5.

### Standard units

Next we'd like to plot the data in standard units. A value in standard units says how many SDs it is
above or below the average:

$$
z = \frac{\text{value} - \text{average}}{\text{SD}}.
$$

Section [14.2](https://dats.columbian.gwu.edu/ds4all/modules/mean.html#standard-units) has a refresher.

---

**Question 1.2.** Fill in `standard_units()`, which converts a vector of numbers to standard units. Use
`mean()` and `sd()`. Then use it inside `mutate()` to make `faithful_su`: the `faithful` tibble with two
new columns, `eruptions_su` and `waiting_su`, holding the durations and the waits in standard units.

In [ ]:
standard_units <- function(numbers) {
  ...
}

faithful_su <- faithful |>
  mutate(
    eruptions_su = ...,
    waiting_su   = ...
  )

faithful_su

In [ ]:
. <- ottr::check("tests/q1_2.R")

Here is the scatter plot again, in standard units:

In [ ]:
ggplot(faithful_su) +
  aes(x = eruptions_su, y = waiting_su) +
  geom_point(alpha = 0.5) +
  labs(
    title = "Old Faithful eruptions, in standard units",
    x = "Eruption duration (standard units)",
    y = "Wait (standard units)"
  ) +
  theme_minimal(base_size = 16)

This plot looks the same as the last one! But the axes are scaled differently, so it's important to
read the tick labels. Each value is now measured in SDs from its average, so 0 on each axis is the
average.

---

**Question 1.3.** Among the following numbers, which would you guess is closest to the correlation
between eruption duration and waiting time? Think about which values a correlation can take.

- `A.` -2
- `B.` -1
- `C.` 0
- `D.` 1
- `E.` 2

Set `r_guess` to the letter, as a string.

Then compute the correlation coefficient `r` from `faithful_su`, as in
[15.1](https://dats.columbian.gwu.edu/ds4all/modules/prediction.html#calculating-r): multiply each pair of standard units, add up the products, and divide by
$n - 1$, where $n$ is the number of eruptions:

$$
r = \frac{\text{sum of (duration in standard units} \times \text{wait in standard units)}}{n - 1}.
$$

In 15.1 you averaged the products, which divides by $n$. We divide by $n - 1$ here because `sd()`,
inside `standard_units()`, divides by $n - 1$ too. With both, $r$ matches R's `cor()` exactly. With 272
eruptions, the two answers are close.

*Hint:* `nrow()` gives the number of rows.

In [ ]:
r_guess <- NULL # YOUR CODE HERE

r <- NULL # YOUR CODE HERE

r

In [ ]:
. <- ottr::check("tests/q1_3.R")

Compare your `r` with R's built-in `cor()`. The two numbers should match:

In [ ]:
cor(faithful$eruptions, faithful$waiting)

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 2. The regression line

Recall that the correlation is the **slope of the regression line when the data are put in standard
units**. The next cell draws the regression line in standard units,

$$
\text{wait in standard units} = r \times \text{duration in standard units},
$$

on top of the data in standard units. `geom_abline()` draws a line with a given slope and intercept.

In [ ]:
ggplot(faithful_su) +
  aes(x = eruptions_su, y = waiting_su) +
  geom_point(alpha = 0.5) +
  geom_abline(slope = r, intercept = 0, color = "red", linewidth = 1) +
  labs(
    title = "The regression line, in standard units",
    x = "Eruption duration (standard units)",
    y = "Wait (standard units)"
  ) +
  theme_minimal(base_size = 16)

How would you take a point in standard units and convert it back to original units? We'd have to
"stretch" its horizontal position by the SD of the durations and its vertical position by the SD of the
waits. That means the same thing would happen to the slope of the line.

Stretching a line horizontally makes it less steep, so we divide the slope by the horizontal stretching
factor. Stretching a line vertically makes it more steep, so we multiply the slope by the vertical
stretching factor:

$$
\text{slope} = r \times \frac{\text{SD of waiting}}{\text{SD of eruptions}}.
$$

The regression line also passes through the point (average duration, average wait), which gives the
intercept:

$$
\text{intercept} = \text{average of waiting} - \text{slope} \times \text{average of eruptions}.
$$

---

**Question 2.1.** Calculate the slope and the intercept of the regression line in original units, and
store them in `slope` and `intercept`. If the stretching explanation doesn't click, see
[15.2](https://dats.columbian.gwu.edu/ds4all/modules/prediction.html#the-equation-of-the-regression-line).

In [ ]:
slope     <- NULL # YOUR CODE HERE
intercept <- NULL # YOUR CODE HERE

c(slope, intercept)

In [ ]:
. <- ottr::check("tests/q2_1.R")

R can also find the regression line for you. `lm()` fits a *linear model*: the formula
`waiting ~ eruptions` means "predict `waiting` from `eruptions`." `coef()` pulls out the line's intercept
and slope. They should match yours:

In [ ]:
faithful_model <- lm(waiting ~ eruptions, data = faithful)

coef(faithful_model)

---

**Question 2.2.** Add the regression line to your scatter plot from Question 1.1, and name the result
`line_plot`. Fill in the blanks so that `geom_abline()` uses your `slope` and `intercept`.

In [ ]:
line_plot <- faithful_scatter +
  geom_abline(slope = ..., intercept = ..., color = "red", linewidth = 1)

line_plot

In [ ]:
. <- ottr::check("tests/q2_2.R")

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 3. Investigating the regression line

The slope and intercept tell you exactly what the regression line looks like. To predict the waiting
time after an eruption, multiply the eruption's duration by `slope` and then add `intercept`.

---

**Question 3.1.** Compute the predicted waiting time after an eruption that lasts 2 minutes, and after
an eruption that lasts 5 minutes. Store them in `two_minute_wait` and `five_minute_wait`.

In [ ]:
two_minute_wait  <- NULL # YOUR CODE HERE
five_minute_wait <- NULL # YOUR CODE HERE

c(two_minute_wait, five_minute_wait)

In [ ]:
. <- ottr::check("tests/q3_1.R")

Both predictions sit on the regression line:

In [ ]:
line_plot +
  annotate("point", x = c(2, 5), y = c(two_minute_wait, five_minute_wait), color = "red", size = 4)

---

**Question 3.2.** Make a prediction for the waiting time after every eruption in `faithful`, in a new
column `predicted_wait`. (Of course, we know exactly what the waiting times were! We are doing this so we
can see how accurate our predictions are.) Then add a column `residual`: the **actual** waiting time
minus the **predicted** waiting time. Name the result `faithful_predictions`. Its first row should look
like this:

| eruptions | waiting | predicted_wait | residual |
|---|---|---|---|
| 3.6 | 79 | 72.10 | 6.90 |

*Hint:* One `mutate()` can make both columns, and the second column can use the first. No loop needed.

Then compute the line's root mean squared error, as in [15.3](https://dats.columbian.gwu.edu/ds4all/modules/prediction.html#root-mean-squared-error): square the
residuals, take their mean, then take the square root. Store it in `faithful_rmse`.

In [ ]:
faithful_predictions <- faithful |>
  mutate(
    predicted_wait = ...,
    residual       = ...
  )

faithful_rmse <- NULL # YOUR CODE HERE

faithful_predictions
faithful_rmse

In [ ]:
. <- ottr::check("tests/q3_2.R")

The RMSE is about 6 minutes, so a typical prediction is off by about 6 minutes, while the waits
themselves run from 43 to 96 minutes.

A useful property of the regression line: its residuals always add up to 0. Yours may be off by a tiny
amount, from rounding. Check:

In [ ]:
sum(faithful_predictions$residual)

Here is a plot of the residuals. Each point is one eruption, and shows how much our prediction
under- or over-estimated its waiting time.

In [ ]:
ggplot(faithful_predictions) +
  aes(x = eruptions, y = residual) +
  geom_point(alpha = 0.5, color = "red") +
  geom_hline(yintercept = 0) +
  labs(
    title = "Residuals from the regression line",
    x = "Eruption duration (minutes)",
    y = "Residual (minutes)"
  ) +
  theme_minimal(base_size = 16)

There's no clear pattern in the residuals, which confirms that it was reasonable to try linear
regression. There are two separate clouds, because the eruption durations fall into two clusters. But
that's a pattern in the durations, not in the relationship between durations and waiting times.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 4. How accurate are different predictions?

Earlier you found that the correlation is close to 1, and the RMSE is small compared with the waits, so
the line fits these eruptions fairly well.

However, unless you have a strong reason to believe that the linear regression model is true, you should
be wary of using it on data very different from the data you fit it to.

---

**Question 4.1.** In `faithful`, no eruption lasted exactly 0, 2.5, or 60 minutes. Using the regression
line, what is the predicted waiting time after an eruption that lasts 0 minutes? 2.5 minutes? An hour?
Store them in `zero_minute_wait`, `two_point_five_minute_wait`, and `hour_wait`.

In [ ]:
zero_minute_wait           <- NULL # YOUR CODE HERE
two_point_five_minute_wait <- NULL # YOUR CODE HERE
hour_wait                  <- NULL # YOUR CODE HERE

c(zero_minute_wait, two_point_five_minute_wait, hour_wait)

Which of these predictions can you trust?

- `A.` Only 2.5 minutes: the data have eruptions near 2.5 minutes, but none near 0 or 60.
- `B.` 0 and 2.5 minutes: both are near the shortest eruptions, and only 60 is far away.
- `C.` All three: with a correlation close to 1, the line predicts well for any duration.
- `D.` None of them: no eruption in the data lasted exactly 0, 2.5, or 60 minutes.

Set `trust_answer` to the letter, as a string.

In [ ]:
trust_answer <- NULL # YOUR CODE HERE

trust_answer

In [ ]:
. <- ottr::check("tests/q4_1.R")

The prediction for 2.5 minutes is believable, since the data have eruptions around that long. A
0-minute eruption isn't an eruption at all, so its predicted wait is meaningless. A 60-minute eruption
might be possible, but we never saw one nearly that long, so it would probably be very different in
character from the ones in `faithful`. Using a line far outside the range of the data it came from is
called **extrapolation**.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 5. Divide and conquer

The scatter plot shows two clusters of eruptions: one with durations around 2 minutes, and another with
durations between 3.5 and 5 minutes. A vertical line at 3 minutes divides them.

In [ ]:
faithful_scatter +
  geom_vline(xintercept = 3, color = "red", linewidth = 1)

Maybe one regression line for each cluster predicts better than one line for all the eruptions.

---

**Question 5.1.** Split `faithful` into two tibbles: `below_3`, the eruptions shorter than 3 minutes,
and `above_3`, the eruptions longer than 3 minutes. No eruption lasted exactly 3 minutes. Then, for each
cluster, compute

- the correlation, with `cor()`: `below_3_r` and `above_3_r`; and
- the slope and intercept of its regression line, the same way as in Question 2.1: `below_3_slope`,
  `below_3_intercept`, `above_3_slope`, and `above_3_intercept`.

In [ ]:
below_3 <- NULL # YOUR CODE HERE
above_3 <- NULL # YOUR CODE HERE

below_3_r <- NULL # YOUR CODE HERE
above_3_r <- NULL # YOUR CODE HERE

below_3_slope     <- NULL # YOUR CODE HERE
below_3_intercept <- NULL # YOUR CODE HERE

above_3_slope     <- NULL # YOUR CODE HERE
above_3_intercept <- NULL # YOUR CODE HERE

tibble(
  cluster   = c("below 3", "above 3"),
  r         = c(below_3_r, above_3_r),
  slope     = c(below_3_slope, above_3_slope),
  intercept = c(below_3_intercept, above_3_intercept)
)

In [ ]:
. <- ottr::check("tests/q5_1.R")

Each cluster's correlation is much weaker than the 0.90 for all the eruptions. Within a cluster, the
duration tells you less about the wait: much of the 0.90 comes from the gap between the clusters.

You just did the calculation from Question 2.1 twice more. In Coding Assignment 5 you'll write functions
that do it for any data.

The plot below shows the two new lines, one for each cluster, along with the line for all the eruptions
in gray.

In [ ]:
faithful_scatter +
  geom_abline(slope = slope, intercept = intercept, color = "gray40", linewidth = 1) +
  annotate("segment", x = 1.5, xend = 3,
           y = below_3_slope * 1.5 + below_3_intercept, yend = below_3_slope * 3 + below_3_intercept,
           color = "red", linewidth = 1) +
  annotate("segment", x = 3, xend = 5.2,
           y = above_3_slope * 3 + above_3_intercept, yend = above_3_slope * 5.2 + above_3_intercept,
           color = "blue", linewidth = 1)

---

**Question 5.2.** Fill in the function `predict_wait()`. It takes durations and returns predicted
waiting times: from the `below_3` line for durations below 3 minutes, and from the `above_3` line
otherwise. It uses `case_when()` instead of `if`, so it works on a whole column at once, inside
`mutate()`.

In [ ]:
predict_wait <- function(duration) {
  case_when(
    duration < 3 ~ ...,
    .default     = ...
  )
}

predict_wait(c(1.5, 2.5, 3.5, 4.5))

The cell below uses `predict_wait()` to predict the wait after every eruption, and plots the
predictions in red with the actual waits.

In [ ]:
faithful_split <- faithful |>
  mutate(
    predicted_wait = predict_wait(eruptions),
    residual       = waiting - predicted_wait
  )

ggplot(faithful_split) +
  aes(x = eruptions) +
  geom_point(aes(y = waiting), alpha = 0.5) +
  geom_point(aes(y = predicted_wait), color = "red") +
  labs(
    title = "Predictions from two regression lines",
    x = "Eruption duration (minutes)",
    y = "Wait until the next eruption (minutes)"
  ) +
  theme_minimal(base_size = 16)

Do you think the predictions from `predict_wait()` are more or less accurate than the predictions from
the single regression line in Part 2?

- `A.` More accurate: each line is fit to its own cluster, so it follows that cluster more closely.
- `B.` Less accurate: each line is fit to fewer eruptions, so its predictions are farther off.
- `C.` Less accurate: the correlation inside each cluster is much weaker than for all the data.
- `D.` About the same: both approaches fit straight lines to the same 272 eruptions in total.

Set `accuracy_answer` to the letter, as a string.

In [ ]:
accuracy_answer <- NULL # YOUR CODE HERE

accuracy_answer

In [ ]:
. <- ottr::check("tests/q5_2.R")

To check, the next cell plots the residuals from one regression line and from two.

In [ ]:
bind_rows(
  `one line`  = faithful_predictions,
  `two lines` = faithful_split,
  .id = "lines"
) |>
  ggplot(aes(x = eruptions, y = residual, color = lines)) +
  geom_point(alpha = 0.5) +
  geom_hline(yintercept = 0) +
  labs(
    title = "Residuals from one line and from two",
    x = "Eruption duration (minutes)",
    y = "Residual (minutes)",
    color = NULL
  ) +
  theme_minimal(base_size = 16) +
  theme(legend.position = "bottom")

And this one compares their RMSEs:

In [ ]:
tibble(
  lines = c("one line", "two lines"),
  rmse  = c(
    sqrt(mean(faithful_predictions$residual^2)),
    sqrt(mean(faithful_split$residual^2))
  )
)

Two lines are a little more accurate: their RMSE is about 5.6 minutes, against about 5.9 for one
line. The residuals from two lines don't have a pattern either, which confirms that linear regression
was also a reasonable choice for each cluster.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


You're done with Lab 6!

Here's what you can now do: check a scatter plot for a linear relationship, compute the correlation
from standard units, find the regression line from $r$, the SDs, and the averages, use it to predict,
measure its errors with residuals and the RMSE, and recognize a prediction that is an extrapolation.

**Important submission information:** Be sure to run the tests and verify that they all pass, then
choose **Save Notebook** from the **File** menu, then Download the .ipynb. Then, go to your course's
Gradescope and submit the corresponding assignment. The name of this assignment is "Lab 06".


### Moral support of this data science course

Good work! Have a rest.

<img src="snooze.jpg" alt="cat sleeping comfortably" width="300"/>

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
# Re-run all the autograder tests in the tests/ folder.
. <- sapply(list.files("tests", pattern = "\\.R$", full.names = TRUE), function(f) ottr::check(f))

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all
images/graphs appear in the output. Click Download in the bar at the top of the notebook. **Please save
before downloading!** Submit your final version to Gradescope.